In [7]:
import langchain
import pinecone
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_ollama import OllamaLLM, OllamaEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pinecone import Pinecone, ServerlessSpec
from langchain_community.vectorstores import Pinecone as PineconeVectorStore
import os

In [8]:
from dotenv import load_dotenv
load_dotenv()

True

In [9]:
def read_docs(directory):
    loader = PyPDFDirectoryLoader(directory)
    documents = loader.load()
    return documents

In [10]:
docs = read_docs('documents/')
len(docs)

140

In [11]:
def chunk_data(docs, chunk_size=1000, chunk_overlap=200):
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size,chunk_overlap=chunk_overlap)
    docs = text_splitter.split_documents(docs)
    return docs

In [12]:
chunk_data(docs)

[Document(metadata={'producer': 'Adobe PDF Library 15.0', 'creator': 'Adobe InDesign CC 13.1 (Windows)', 'creationdate': '2018-10-16T13:23:57+02:00', 'moddate': '2018-10-16T13:34:23+02:00', 'trapped': '/False', 'source': 'documents/FIFA+1.pdf', 'total_pages': 140, 'page': 0, 'page_label': '1'}, page_content='TECHNICAL  \nREPORT\n2018 FIFA WORLD CUP RUSSIATM'),
 Document(metadata={'producer': 'Adobe PDF Library 15.0', 'creator': 'Adobe InDesign CC 13.1 (Windows)', 'creationdate': '2018-10-16T13:23:57+02:00', 'moddate': '2018-10-16T13:34:23+02:00', 'trapped': '/False', 'source': 'documents/FIFA+1.pdf', 'total_pages': 140, 'page': 1, 'page_label': '2'}, page_content='2\n2018 FIFA World Cup RussiaTM\nTABLE OF CONTENTS\nPart 1: Foreword 3\nPart 2: Technical Study Group 5\nPart 3: Tournament observation 6\nPart 4: Team profiles 15\nPart 5: Performance insights 80\nPart 6: FIFA awards 103\nPart 7: Match information 106\nPart 8: Technical Study Group support 140\nPlease note that to view any a

In [13]:
embeddings = OllamaEmbeddings(model='bge-m3')

In [14]:
pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))

index_name = "fifa2018vector"

# check if index exists
if index_name not in pc.list_indexes().names():
    pc.create_index(
        name=index_name,
        dimension=1024,
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        )
    )

index = pc.Index(index_name)

In [15]:
vectorstore = PineconeVectorStore.from_documents(docs,embeddings,index_name=index_name)

In [16]:
#Cosine Similarity Retrieve Results
def retrieve_query(query, k=2):
    matching_results = vectorstore.similarity_search(query, k=2)
    return matching_results

In [18]:
from langchain_classic.chains.question_answering import load_qa_chain

In [19]:
llm = OllamaLLM(model='llama2', temperature=0.5)
chain = load_qa_chain(llm, chain_type='stuff')

/var/folders/qv/7r3vg1ns6qbgvs0t3_dxc3qm0000gn/T/ipykernel_32235/465001491.py:2: LangChainDeprecationWarning: This class is deprecated. See the following migration guides for replacements based on `chain_type`:
stuff: https://python.langchain.com/docs/versions/migrating_chains/stuff_docs_chain
map_reduce: https://python.langchain.com/docs/versions/migrating_chains/map_reduce_chain
refine: https://python.langchain.com/docs/versions/migrating_chains/refine_chain
map_rerank: https://python.langchain.com/docs/versions/migrating_chains/map_rerank_docs_chain

See also guides on retrieval and question-answering here: https://python.langchain.com/docs/how_to/#qa-with-rag
  chain = load_qa_chain(llm, chain_type='stuff')


In [20]:
# Search answers from VectorDB
def retrieve_answers(query):
    doc_search = retrieve_query(query)
    print(doc_search)
    response = chain.invoke({
        "input_documents": doc_search,
        "question": query
    })
    return response["output_text"]

In [22]:
our_query = "Portugal team members name"
answer = retrieve_answers(our_query)
print(answer)

[Document(metadata={'creationdate': '2018-10-16T13:23:57+02:00', 'creator': 'Adobe InDesign CC 13.1 (Windows)', 'moddate': '2018-10-16T13:34:23+02:00', 'page': 14.0, 'page_label': '15', 'producer': 'Adobe PDF Library 15.0', 'source': 'documents/FIFA+1.pdf', 'total_pages': 140.0, 'trapped': '/False'}, page_content='15\n2018 FIFA World Cup RussiaTM\nTEAM PROFILES'), Document(metadata={'creationdate': '2018-10-16T13:23:57+02:00', 'creator': 'Adobe InDesign CC 13.1 (Windows)', 'moddate': '2018-10-16T13:34:23+02:00', 'page': 110.0, 'page_label': '111', 'producer': 'Adobe PDF Library 15.0', 'source': 'documents/FIFA+1.pdf', 'total_pages': 140.0, 'trapped': '/False'}, page_content="111\n2018 FIFA World Cup RussiaTM\nCristiano Ronaldo was the star man for Portugal at the 2018 FIFA \nWorld Cup™ for the second match running, scoring the winner \nagainst Morocco at the Luzhniki Stadium to eliminate the North \nAfricans. After his hat-trick heroics against Spain in his side’s \nopener claimed a si